# ActionShap — complete end-to-end runner

This is the canonical **Run All** notebook for revision 4. It performs the entire reproducible workflow from an empty local data directory:

1. resolves the repository and installs the locked recommendation environment;
2. downloads MovieLens-1M and Amazon Digital Music from their declared sources;
3. verifies/prepares content-addressed datasets;
4. loads and audits both temporal splits;
5. inspects the frozen real-data preflight;
6. runs all independent convergence studies;
7. runs the two-dataset, two-model, five-seed primary/full-catalogue matrix;
8. runs every predeclared sensitivity;
9. creates hierarchical statistics, tables, figures, manifests, and the result archive;
10. validates the manuscript and displays final status.

**Expected cost:** the final suite is intentionally expensive and may take hours. Raw datasets, environments, and raw JSON are ignored by Git. The notebook never turns a failed gate or incomplete matrix into a paper claim.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys

# Works when opened from the repository root or from ActionShap/code.
START = Path.cwd().resolve()
CANDIDATES = [
    START,
    START / "paper-ideas" / "ActionShap" / "code",
]
CODE_ROOT = next(
    (candidate for candidate in CANDIDATES if (candidate / "scripts" / "run_final_suite.py").exists()),
    None,
)
if CODE_ROOT is None:
    raise RuntimeError(
        "Cannot locate paper-ideas/ActionShap/code. Open the notebook from the "
        "repository root or from its code directory."
    )
ACTIONSHAP_ROOT = CODE_ROOT.parent
REPO_ROOT = ACTIONSHAP_ROOT.parent.parent
CONFIG_PATH = CODE_ROOT / "configs" / "final.yaml"
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

# Run-All controls. The defaults execute the full requested workflow.
INSTALL_DEPENDENCIES = True
DOWNLOAD_DATA_IF_MISSING = True
ACCEPT_DATASET_TERMS = True  # Set only after reviewing GroupLens and Amazon terms/citations.
FORCE_REDOWNLOAD = False
DOWNLOAD_ATTEMPTS = 2
DOWNLOAD_TIMEOUT_SECONDS = 30
# Optional manual fallbacks when institutional networking blocks UCSD/GroupLens.
# Point these at the unmodified source archives, or leave them empty.
MOVIELENS_LOCAL_ARCHIVE = ""
AMAZON_LOCAL_SOURCE = ""
RUN_FINAL_SUITE = True

DATA_SETUP_OK = None
DATA_SETUP_ERROR = None
PIPELINE_OK = None
PIPELINE_ERROR = None

print("REPO_ROOT       =", REPO_ROOT)
print("CODE_ROOT       =", CODE_ROOT)
print("CONFIG_PATH     =", CONFIG_PATH)
print("Python          =", sys.version.split()[0])


## 1. Install the locked environment

The lock file is the exact tested Python environment. Installation is idempotent. If a platform cannot resolve a locked wheel, set `INSTALL_DEPENDENCIES=False`, install `requirements-recommendation.txt` manually, and preserve the resolved versions in result provenance.


In [ ]:
if INSTALL_DEPENDENCIES:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-r",
            str(CODE_ROOT / "requirements-recommendation.lock"),
        ],
        cwd=CODE_ROOT,
        check=True,
    )
else:
    print("Dependency installation skipped by configuration.")


In [ ]:
# Import only after dependency installation.
import numpy as np
import pandas as pd
import scipy
import sklearn
import yaml

from actionshap.recommendation_data import (
    load_interactions_csv,
    load_movielens_1m,
)

CONFIG = yaml.safe_load(CONFIG_PATH.read_text())
print(
    {
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scipy": scipy.__version__,
        "scikit-learn": sklearn.__version__,
    }
)
print(yaml.safe_dump(CONFIG, sort_keys=False))


## 2. Download and prepare both datasets

The downloader tries bounded retries across declared mirrors:

- GroupLens MovieLens-1M; extracted `ratings.dat` must match the pinned preflight SHA-256.
- Amazon Review Data (2018), `Digital_Music_5.json.gz`; the builder retains ratings at least 4, resolves duplicates deterministically, reapplies iterative 5-core filtering, and writes source/output hashes.

If all mirrors are blocked, download the unmodified files with a browser and set `MOVIELENS_LOCAL_ARCHIVE` or `AMAZON_LOCAL_SOURCE` in the configuration cell. Expected manual Amazon destination:

```text
paper-ideas/ActionShap/code/data/amazon-digital-music/Digital_Music_5.json.gz
```

The cell records failure without cascading into unrelated notebook tracebacks. Rerun it after providing the local source.


In [ ]:
DATA_SETUP_OK = True
DATA_SETUP_ERROR = None

if DOWNLOAD_DATA_IF_MISSING:
    if not ACCEPT_DATASET_TERMS:
        DATA_SETUP_OK = False
        DATA_SETUP_ERROR = (
            "Review the GroupLens and Amazon dataset terms, then set "
            "ACCEPT_DATASET_TERMS=True."
        )
    else:
        command = [
            sys.executable,
            str(CODE_ROOT / "scripts" / "download_datasets.py"),
            "--dataset",
            "all",
            "--accept-dataset-terms",
            "--attempts",
            str(DOWNLOAD_ATTEMPTS),
            "--timeout",
            str(DOWNLOAD_TIMEOUT_SECONDS),
        ]
        if FORCE_REDOWNLOAD:
            command.append("--force")
        if MOVIELENS_LOCAL_ARCHIVE:
            command.extend(["--movielens-local-archive", MOVIELENS_LOCAL_ARCHIVE])
        if AMAZON_LOCAL_SOURCE:
            command.extend(["--amazon-local-source", AMAZON_LOCAL_SOURCE])
        completed = subprocess.run(
            command,
            cwd=CODE_ROOT,
            text=True,
            capture_output=True,
            check=False,
        )
        if completed.stdout:
            print(completed.stdout)
        if completed.returncode != 0:
            DATA_SETUP_OK = False
            DATA_SETUP_ERROR = completed.stderr.strip() or (
                f"dataset setup exited with status {completed.returncode}"
            )
else:
    print("Dataset download skipped; existing files will be audited next.")

if not DATA_SETUP_OK:
    print("\n" + "=" * 88)
    print("DATA SETUP INCOMPLETE — LATER EXPERIMENT CELLS WILL BE SKIPPED")
    print("=" * 88)
    print(DATA_SETUP_ERROR[-5000:] if DATA_SETUP_ERROR else "Unknown setup error")
    print("\nRecovery options:")
    print("1. Rerun this cell; alternate UCSD mirrors are tried automatically.")
    print("2. Download Digital_Music_5.json.gz in a browser and set AMAZON_LOCAL_SOURCE.")
    print(
        "3. Or place it at",
        CODE_ROOT / "data" / "amazon-digital-music" / "Digital_Music_5.json.gz",
        "and rerun this cell.",
    )
else:
    print("Dataset download/preparation completed successfully.")


## 3. Load and audit temporal data

This cell proves that both prepared files are readable under the exact final configuration. It reports post-filter users/items/interactions, history lengths, hashes, and deterministic split reconstruction. It does **not** inspect explanation outcomes.


In [ ]:
def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def load_configured_dataset(spec):
    source = (CODE_ROOT / spec["path"]).resolve()
    if not source.exists():
        raise FileNotFoundError(source)
    if spec["format"] == "ml1m":
        data = load_movielens_1m(
            source,
            rating_threshold=float(spec.get("rating_threshold", 4.0)),
            minimum_interactions=int(CONFIG["minimum_interactions"]),
        )
        repeat = load_movielens_1m(
            source,
            rating_threshold=float(spec.get("rating_threshold", 4.0)),
            minimum_interactions=int(CONFIG["minimum_interactions"]),
        )
    else:
        kwargs = {
            "user_column": spec.get("user_column", "user"),
            "item_column": spec.get("item_column", "item"),
            "timestamp_column": spec.get("timestamp_column", "timestamp"),
            "rating_column": spec.get("rating_column", "rating"),
            "rating_threshold": float(spec.get("rating_threshold", 4.0)),
            "minimum_interactions": int(CONFIG["minimum_interactions"]),
        }
        data = load_interactions_csv(source, **kwargs)
        repeat = load_interactions_csv(source, **kwargs)
    if data.test != repeat.test or data.validation != repeat.validation:
        raise AssertionError(f"Temporal split is not deterministic for {spec['name']}")
    lengths = np.array([len(history) for history in data.train.values()])
    return data, {
        "dataset": spec["name"],
        "path": str(source.relative_to(REPO_ROOT)),
        "sha256": file_sha256(source),
        "users": len(data.test),
        "items": data.n_items,
        "interactions_after_filter": int(lengths.sum() + 2 * len(lengths)),
        "train_history_median": float(np.median(lengths)),
        "train_history_p95": float(np.quantile(lengths, 0.95)),
        "minimum_train_history": int(lengths.min()),
        "deterministic_split": True,
    }


LOADED_DATASETS = {}
audit_rows = []

if not DATA_SETUP_OK:
    DATA_AUDIT = pd.DataFrame()
    print("Data audit skipped because dataset setup is incomplete.")
else:
    try:
        for dataset_spec in CONFIG["datasets"]:
            data, audit = load_configured_dataset(dataset_spec)
            LOADED_DATASETS[dataset_spec["name"]] = data
            audit_rows.append(audit)
        DATA_AUDIT = pd.DataFrame(audit_rows)
        display(DATA_AUDIT)
        assert len(LOADED_DATASETS) == 2
        assert all(row["minimum_train_history"] >= 2 for row in audit_rows)
    except Exception as error:
        DATA_SETUP_OK = False
        DATA_SETUP_ERROR = f"data audit failed: {error}"
        DATA_AUDIT = pd.DataFrame()
        print(DATA_SETUP_ERROR)


## 4. Inspect the frozen design preflight

The preflight selected primary ItemKNN, `n_max=20`, target-margin attribution, and the conservative final floor `M=500` before the final result matrix. It is content-addressed and excluded from headline result aggregation.


In [ ]:
PREFLIGHT = None
if not DATA_SETUP_OK:
    print("Preflight audit skipped because dataset setup is incomplete.")
else:
    preflight_path = (
        ACTIONSHAP_ROOT / "paper" / "preflight" / "movielens_masking_gate.json"
    )
    PREFLIGHT = json.loads(preflight_path.read_text())
    movielens_hash = DATA_AUDIT.loc[
        DATA_AUDIT["dataset"] == "MovieLens-1M", "sha256"
    ].iloc[0]
    if PREFLIGHT["source_sha256"] != movielens_hash:
        DATA_SETUP_OK = False
        DATA_SETUP_ERROR = "MovieLens payload differs from the frozen design preflight"
        print(DATA_SETUP_ERROR)
    else:
        gate_frame = pd.DataFrame(PREFLIGHT["final_window_gate_runs"])
        display(gate_frame)
        print(PREFLIGHT["decision"])
        print(PREFLIGHT["utility_decision"])
        assert gate_frame.loc[gate_frame["model"] == "itemknn", "passed"].all()
        assert PREFLIGHT["utility_convergence_preflight"]["target_margin"][
            "selected_permutations"
        ] == 250


## 5. Execute the complete final suite

The suite runs **85 scientific commands** plus asset/manuscript/package validation. It stops on any blocking primary gate, missing experiment, inconsistent cohort, inadequate target-margin convergence, or primary-quality failure. Robustness-model and NDCG-utility failures remain visible as bounded findings.

Set `RUN_FINAL_SUITE=False` only when you want to audit already-generated schema-v2 raw files without rerunning models.


In [ ]:
PIPELINE_OK = bool(DATA_SETUP_OK)
PIPELINE_ERROR = None

if not DATA_SETUP_OK:
    print("Final suite skipped because dataset setup is incomplete.")
elif RUN_FINAL_SUITE:
    completed = subprocess.run(
        [
            sys.executable,
            str(CODE_ROOT / "scripts" / "run_final_suite.py"),
            "--config",
            str(CONFIG_PATH),
        ],
        cwd=CODE_ROOT,
        check=False,
    )
    if completed.returncode != 0:
        PIPELINE_OK = False
        PIPELINE_ERROR = f"final suite exited with status {completed.returncode}"
        print(PIPELINE_ERROR)
else:
    print("Final suite execution skipped; validating existing outputs only.")
    completed = subprocess.run(
        [sys.executable, str(CODE_ROOT / "scripts" / "make_paper_assets.py")],
        cwd=CODE_ROOT,
        check=False,
    )
    if completed.returncode != 0:
        PIPELINE_OK = False
        PIPELINE_ERROR = f"asset generation exited with status {completed.returncode}"
        print(PIPELINE_ERROR)


## 6. Inspect final validation and headline assets

Only `status: PASS` permits numerical claims. The manuscript intentionally keeps result placeholders until the final tables exist and authors write conclusions that match the validated data.


In [ ]:
validation_path = (
    ACTIONSHAP_ROOT / "paper" / "final" / "manifests" / "validation_report.json"
)
manifest_path = (
    ACTIONSHAP_ROOT / "paper" / "final" / "manifests" / "asset_manifest.json"
)

if not DATA_SETUP_OK:
    VALIDATION = {"status": "DATA_SETUP_INCOMPLETE", "error": DATA_SETUP_ERROR}
    print(json.dumps(VALIDATION, indent=2))
elif not validation_path.exists():
    VALIDATION = {"status": "MISSING", "error": PIPELINE_ERROR}
    print(json.dumps(VALIDATION, indent=2))
else:
    VALIDATION = json.loads(validation_path.read_text())
    print(json.dumps(VALIDATION, indent=2))
    if VALIDATION["status"] == "PASS" and manifest_path.exists():
        MANIFEST = json.loads(manifest_path.read_text())
        print("Source files:", len(MANIFEST["source_files"]))
        print("Generated files:", len(MANIFEST["generated_files"]))
        method_table = (
            ACTIONSHAP_ROOT / "paper" / "final" / "tables" / "method_metrics.csv"
        )
        paired_table = (
            ACTIONSHAP_ROOT / "paper" / "final" / "tables" / "paired_tests.csv"
        )
        display(pd.read_csv(method_table).head(30))
        display(pd.read_csv(paired_table).head(30))
    else:
        print("Final numerical claims remain blocked; inspect errors above.")


## 7. Validate manuscript and release package

Static validation checks citations, environments, pilot-value contamination, generated tables, final asset status, and remaining placeholders. `--require-final` is intentionally not used here until authors replace the result prose placeholders.


In [ ]:
subprocess.run(
    [sys.executable, str(CODE_ROOT / "scripts" / "validate_manuscript.py")],
    cwd=CODE_ROOT,
    check=False,
)

release_root = CODE_ROOT / "results" / "release"
archives = sorted(release_root.glob("actionshap-schema-v2-results.tar.gz*"))
print("Release artifacts:")
if archives:
    for artifact in archives:
        print(" -", artifact, artifact.stat().st_size, "bytes")
else:
    print(" - none yet")

print("\nNotebook terminal status:", VALIDATION.get("status", "UNKNOWN"))
if DATA_SETUP_ERROR:
    print("Data setup message:", DATA_SETUP_ERROR)
if PIPELINE_ERROR:
    print("Pipeline message:", PIPELINE_ERROR)


## Completion checklist

A successful Run All ends with:

- two audited datasets and matching SHA-256 provenance;
- primary ItemKNN gates passing for every dataset/seed;
- target-margin convergence and an explicitly reported NDCG stress test;
- exact dual-utility `B<=2` oracles for all primary users;
- five methods on matched cohorts across all required conditions;
- `paper/final/manifests/validation_report.json` equal to `PASS`;
- generated tables/figures and a content-addressed raw-result archive;
- manuscript static validation with only result-writing placeholders remaining.

After writing the numerical Results, Discussion, Abstract, and Conclusion, run:

```bash
python scripts/validate_manuscript.py --require-final
```
